In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv(
    "./data/etf_prices_daily.csv",
    index_col=0,
    parse_dates=True
)

monthly_prices = prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change()

In [2]:
def run_momentum_backtest(
    monthly_prices,
    top_n=3,
    lookback_months=12,
    skip_months=1,
    cost_per_turnover=0.001
):
    monthly_returns = monthly_prices.pct_change()

    signal = (
        monthly_prices.shift(skip_months)
        / monthly_prices.shift(lookback_months)
    ) - 1

    weights = pd.DataFrame(
        0.0,
        index=signal.index,
        columns=signal.columns
    )

    for date in signal.index:
        scores = signal.loc[date].dropna()

        if len(scores) >= top_n:
            selected = scores.nlargest(top_n).index
            weights.loc[date, selected] = 1 / top_n

    gross_returns = (
        weights.shift(1) * monthly_returns
    ).sum(axis=1)

    turnover = weights.diff().abs().sum(axis=1) / 2
    costs = turnover * cost_per_turnover

    net_returns = gross_returns - costs

    return net_returns.dropna(), weights, turnover

In [3]:
def quick_metrics(returns):
    equity = (1 + returns).cumprod()
    years = len(returns) / 12

    annualised_return = equity.iloc[-1] ** (1 / years) - 1
    annualised_volatility = returns.std() * np.sqrt(12)
    sharpe = (
        returns.mean() / returns.std()
    ) * np.sqrt(12)

    max_drawdown = (
        equity / equity.cummax() - 1
    ).min()

    return pd.Series({
        "Annualised Return": annualised_return,
        "Annualised Volatility": annualised_volatility,
        "Sharpe Ratio": sharpe,
        "Maximum Drawdown": max_drawdown
    })

holding_results = {}

for top_n in [2, 3, 4]:
    net_returns, _, _ = run_momentum_backtest(
        monthly_prices=monthly_prices,
        top_n=top_n,
        lookback_months=12,
        skip_months=1,
        cost_per_turnover=0.001
    )

    holding_results[f"Top {top_n}"] = quick_metrics(net_returns)

holding_metrics = pd.DataFrame(holding_results)

holding_metrics

,Top 2,Top 3,Top 4
Annualised Return,0.098338,0.113827,0.114269
Annualised Volatility,0.131839,0.129548,0.128978
Sharpe Ratio,0.780044,0.900289,0.906632
Maximum Drawdown,-0.266099,-0.248452,-0.261843


In [13]:
lookback_results = {}

for lookback in [6, 9, 12]:
    net_returns, _, _ = run_momentum_backtest(
        monthly_prices=monthly_prices,
        top_n=3,
        lookback_months=lookback,
        skip_months=1,
        cost_per_turnover=0.001
    )

    lookback_results[f"{lookback}-1 Momentum"] = quick_metrics(net_returns)

lookback_metrics = pd.DataFrame(lookback_results)

lookback_metrics

,6-1 Momentum,9-1 Momentum,12-1 Momentum
Annualised Return,0.096514,0.119990,0.113827
Annualised Volatility,0.142112,0.127772,0.129548
Sharpe Ratio,0.721624,0.954607,0.900289
Maximum Drawdown,-0.258810,-0.248725,-0.248452


In [6]:
cost_results = {}

for cost_bps in [5, 10, 25]:
    net_returns, _, _ = run_momentum_backtest(
        monthly_prices=monthly_prices,
        top_n=3,
        lookback_months=12,
        skip_months=1,
        cost_per_turnover=cost_bps / 10_000
    )

    cost_results[f"{cost_bps} bps"] = quick_metrics(net_returns)

cost_metrics = pd.DataFrame(cost_results)

cost_metrics

,5 bps,10 bps,25 bps
Annualised Return,0.114437,0.113827,0.111999
Annualised Volatility,0.129565,0.129548,0.129500
Sharpe Ratio,0.904444,0.900289,0.887786
Maximum Drawdown,-0.248332,-0.248452,-0.248810


In [15]:
holding_metrics.to_csv("./data/robustness_top_n.csv")
lookback_metrics.to_csv("./data/robustness_lookback.csv")
cost_metrics.to_csv("./data/robustness_costs.csv")

print("Robustness checks saved.")

Robustness checks saved.
